# Does the gate pick better frames for a VLM? EgoEverything, all 12 overlapping videos

[EgoEverything](https://huggingface.co/datasets/roxqtang/EgoEverything) (arXiv 2604.08342)
asks 5-option questions about what an AR-glasses wearer saw, built from real gaze traces on
Aria Everyday Activities (AEA). Each question names a **keyframe time** where the answer is
visible.

**The test:** at the **same number of frames**, does the gate's choice of frames give the VLM
better answers than spreading them evenly?

## What changed since the pilot

The pilot used only the 4 test videos that have questions (204 questions) and found
**policy D 39.2% vs fixed interval 43.6%**: D clusters frames at settle moments and leaves
gaps, so the moment a question asks about was often not sent. This run:

- uses **all 12 gate videos that have EgoEverything questions (834 questions)**. The 8 that
  were training videos are scored by the fold model that held each one out, so every
  prediction still comes from a model that never trained on that video;
- adds a **hybrid**: half the frames evenly spaced, the other half from the gate, placed in
  the gaps the even spacing leaves;
- adds **policy E** (D plus one frame every 5 s of motion), against fixed interval at E's
  own budget.

Pilot results stay in their own file; this run writes to a new one.

| condition | frames given to the VLM (all from the same 1 fps frames as the labels) |
|---|---|
| **hybrid 50/50 (D budget)** | half evenly spaced, half the gate's settle/refix frames that fill the largest gaps |
| **uniform (D budget)** | evenly spaced, as many per video as D: full: **the key baseline** |
| D: full | policy D (send when the view settles + on each REFIXATION), full model |
| blind (no frames) | question and options only (chance is 20%) |
| E: full | policy D + one frame every 5 s during long motion |
| uniform (E budget) | evenly spaced, as many per video as E: full |
| D: ground truth | policy D on the DINOv2 labels: the policy's ceiling |
| random (D budget) | as many as D, at random |

**Runtime:** A100 recommended (~1.5 s per question, ~20 min per condition, ~2.5 h for all
eight; the first four take ~1 h). Results are written to Drive as they go; a rerun resumes.

## 1. Clone, mount, install

In [ ]:
import os, sys, json, subprocess

REPO, BRANCH = '/content/GazeVLM', 'eyehead-gate'     # 'main' once merged
URL = 'https://github.com/shubhamOjha1000/GazeVLM-HWSW-Codesign.git'
if os.path.exists(os.path.join(REPO, '.git')):
    subprocess.run(['git', '-C', REPO, 'fetch', '-q', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO, 'checkout', '-q', '-B', BRANCH, f'origin/{BRANCH}'], check=True)
else:
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH, URL, REPO], check=True)
os.chdir(REPO)
sys.path.insert(0, REPO)
print(subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)
assert os.path.exists('vlmeval/qwen_mcq.py'), f'vlmeval/ is not on branch {BRANCH!r}'

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'qwen-vl-utils', 'huggingface_hub'],
               check=True)

import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt

from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/GazeVLM'
FOLDS     = os.path.join(DRIVE_DIR, 'folds')
RUNS      = os.path.join(DRIVE_DIR, 'runs')
VLM_DIR   = os.path.join(DRIVE_DIR, 'vlm', 'egoeverything')
FRAMES_TAR = os.path.join(VLM_DIR, 'frames_tar')          # one .tar per video, reused later
LOCAL_FRAMES = '/content/frames_1fps'
os.makedirs(VLM_DIR, exist_ok=True)

CKPT_FULL = os.path.join(RUNS, 'eyehead')                  # the full model's 5 fold checkpoints
missing = [i for i in range(1, 6) if not os.path.exists(os.path.join(CKPT_FULL, f'fold{i}.pt'))]
assert not missing, f'missing fold checkpoints {missing} in {CKPT_FULL} -- run the eyehead notebook first'
print('full model checkpoints: 5 folds found in', CKPT_FULL)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')

## 2. The AEA download-links JSON

The signed links expire after ~14 days. This cell checks the copy on Drive:

- **missing or expired** → it asks you to upload the fresh `aea_download_urls.json`, checks
  its links work, and **replaces the older copies** on Drive (`MyDrive/GazeVLM/` and, if
  present, `MyDrive/aea/`, so every notebook picks up the fresh one)
- **still valid** → nothing to do

Set `FORCE_UPLOAD = True` to replace it anyway.

In [ ]:
from vlmeval.frames import links_fresh, install_links

FORCE_UPLOAD = False
LINKS = os.path.join(DRIVE_DIR, 'aea_download_urls.json')
OLD_COPIES = [LINKS] + [p for p in ['/content/drive/MyDrive/aea/aea_download_urls.json']
                        if os.path.exists(p)]

ok, msg = links_fresh(LINKS) if os.path.exists(LINKS) else (False, 'not on Drive')
print('Drive copy:', msg)
if FORCE_UPLOAD or not ok:
    from google.colab import files
    print('upload the fresh aea_download_urls.json ...')
    up = files.upload()
    name = list(up.keys())[0]
    ok, msg = links_fresh(name)
    print('uploaded file:', msg)
    assert ok, 'the uploaded JSON does not work either -- download a fresh one from projectaria.com/datasets/aea'
    install_links(name, OLD_COPIES)
    os.remove(name)                      # the copies live on Drive; keep the repo clean
    for p in OLD_COPIES:
        print('   replaced', p)
links = json.load(open(LINKS))
print(f"using {LINKS}: {len(links['sequences'])} sequences")
print('(the eyehead notebook\'s IMU calibration, section 3, can now download its VRS too)')

## 3. EgoEverything questions on the gate's videos

In [ ]:
from vlmeval import egoev

qs_all = egoev.load_questions()
aea = qs_all[qs_all.video.str.startswith('loc')]
clips_all, test = egoev.gate_clips(FOLDS, test_only=False)
gate_videos = {c['sequence'] for c in clips_all}
test_videos = {c['sequence'] for c in clips_all if c['is_test']}

qs = aea[aea.video.isin(gate_videos)].reset_index(drop=True)
clips = [c for c in clips_all if c['sequence'] in set(qs.video)]
videos = [c['sequence'] for c in clips]
n_frames = {c['sequence']: len(c['frame_no']) for c in clips}
print(f'EgoEverything: {len(qs_all):,} questions, {len(aea):,} on AEA ({aea.video.nunique()} videos)')
print(f'on the gate videos: {len(qs):,} questions on {len(videos)} videos '
      f'({qs.video.isin(test_videos).sum()} questions on {len(set(videos) & test_videos)} test videos)\n')

per = qs.groupby('video').agg(questions=('qid', 'size'), t0_max=('t0', 'max'),
                              median_dist2gaze=('dist2gaze', 'median'))
per['video_s'] = [n_frames[v] for v in per.index]
per['split'] = ['test' if v in test_videos else 'train (held out by its fold)' for v in per.index]
display(per.sort_values('questions', ascending=False).round(1))
print('answer letters:', qs.answer.value_counts().sort_index().to_dict())

## 4. Held-out gate predictions, and which frames each condition sends

Each training video is scored by the fold model that held it out for validation; test
videos by the 5-fold ensemble. The table is cached on Drive. The quick accuracy check below
should be close to the cross-validation numbers (~0.6 four-class accuracy), not higher,
which would mean a video leaked into its own training.

In [ ]:
from eyehead.oof import oof_predictions

OOF = os.path.join(VLM_DIR, 'oof_preds_full.csv')
if os.path.exists(OOF):
    oof = pd.read_csv(OOF)
    print('loaded', OOF)
else:
    oof = oof_predictions(FOLDS, CKPT_FULL)
    oof.to_csv(OOF, index=False)
    print('wrote', OOF)

o12 = oof[oof.sequence.isin(videos)]
print(f"\nheld-out 4-class accuracy on these {len(videos)} videos: "
      f"{(o12.pred == o12.quad).mean():.3f}   by source:")
display(o12.assign(correct=o12.pred == o12.quad).groupby('source').correct.agg(['mean', 'size']).round(3))

labels = {'full': egoev.video_labels(clips, oof)}
sel = egoev.select_conditions(clips, labels, main='full', timeout=5, hybrid_frac=0.5, seed=0)
json.dump(sel, open(os.path.join(VLM_DIR, 'selection_all12.json'), 'w'))
display(egoev.budget_table(sel, n_frames).round(2))

# per video: frames each condition sends; D_interval_s = the spacing the per-video-matched
# uniform used, which a deployed fixed interval cannot adapt to
pv_budget = pd.DataFrame({c: {v: len(sel[c][v]) for v in videos}
                          for c in ('D: full', 'fixed every 5 s', 'fixed every 6 s')})
pv_budget.insert(0, 'video_s', [n_frames[v] for v in pv_budget.index])
pv_budget['D_interval_s'] = (pv_budget['video_s'] / pv_budget['D: full']).round(1)
display(pv_budget.sort_values('D_interval_s'))
print(f"D: full sends one frame every {pv_budget.video_s.sum() / pv_budget['D: full'].sum():.1f} s on average")

## 5. Frames: download once, cache on Drive

The 4 test videos are already cached from the pilot; the 8 others are downloaded once
(~165 MB each).

In [ ]:
from vlmeval.frames import ensure_frames, frame_path

SIZE = 448
metas = {}
for v in videos:
    metas[v] = ensure_frames(v, links, LOCAL_FRAMES, FRAMES_TAR, size=SIZE)
    m = metas[v]
    flag = 'OK' if m['n'] >= n_frames[v] else f'!! only {m["n"]} frames, labels need {n_frames[v]}'
    print(f"   {v:<26} {m['n']:4d} frames  {flag}")

## 6. Alignment check and target coverage

**Alignment:** the frame nearest each question's keyframe, with its box. The boxes should sit
on the named objects.

**Target coverage:** distance from each question's keyframe to the nearest sent frame. The
pilot showed this decides the score: D covered 40% of keyframes within 2 s against 66% for
fixed interval. The hybrid should close most of that gap.

In [ ]:
import matplotlib.patches as mpatches

def frame_time(v, k):
    return metas[v]['ts'][k] if k < metas[v]['n'] else k

show = qs[qs.bbox.notna() & qs.t0.notna()].sample(min(6, len(qs)), random_state=1)
fig, ax = plt.subplots(1, len(show), figsize=(3.4 * len(show), 3.8))
for a_, q in zip(np.atleast_1d(ax), show.itertuples()):
    m = metas[q.video]
    k = int(np.argmin(np.abs(np.array(m['ts']) - q.t0)))
    img = plt.imread(frame_path(m, k))
    s = img.shape[1] / m['orig_w']
    x1, y1, x2, y2 = [b * s for b in q.bbox]
    a_.imshow(img)
    a_.add_patch(mpatches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, ec='yellow', lw=2))
    a_.set_title(f"{q.object}\nt0={q.t0:.1f}s  frame {k}", fontsize=8); a_.axis('off')
plt.tight_layout(); plt.show()

cov = egoev.coverage(qs, sel, frame_time)
ctab = cov.groupby('condition', sort=False).agg(
    questions=('qid', 'size'), median_dist_s=('dist', 'median'),
    **{f'within_{t}': (f'hit_{t}', 'mean') for t in ('1s', '2s', '5s')})
for c in ('within_1s', 'within_2s', 'within_5s'):
    ctab[c] *= 100
display(ctab.round(1))

## 7. Load Qwen2.5-VL-3B and smoke test

Same model and resolution as the pilot (`448*448`, about 256 visual tokens per frame).

In [ ]:
from vlmeval import qwen_mcq
import time

MODEL_ID, MAX_PIXELS = 'Qwen/Qwen2.5-VL-3B-Instruct', 448 * 448
model, proc, letter_ids, dtype = qwen_mcq.load_model(MODEL_ID)
print(MODEL_ID, dtype, f'{torch.cuda.memory_allocated()/1e9:.1f} GB on GPU')

q = qs.iloc[0]
frames = [(frame_path(metas[q.video], k), metas[q.video]['ts'][k])
          for k in sel['hybrid 50/50 (D budget)'][q.video]]
t0 = time.time()
p, ntok = qwen_mcq.score(model, proc, letter_ids,
                         qwen_mcq.build_messages(q.question, q.options, frames, MAX_PIXELS))
print(f'{len(frames)} frames, {ntok} prompt tokens, {time.time()-t0:.1f} s')
print('Q:', q.question)
for L, o, pi in zip('ABCDE', q.options, p):
    print(f"   {L}. {o:<60} p={pi:.3f}{'   <- answer' if L == q.answer else ''}")
assert np.isfinite(p).all(), 'NaN logits: use an L4/A100 (bf16) or lower MAX_PIXELS'

## 8. Score the conditions

**Quick test (default):** `QUICK = True` scores only the 4 test videos (204 questions) with
**fixed every 3 s** and **D: full**. D: full was already scored on these videos in the
previous run, so only the fixed interval runs (~5 min on an A100). Note the budgets differ:
every 3 s sends about twice as many frames as D, so this asks whether a dense fixed
interval does better, not an equal-cost comparison.

**Full run:** `QUICK = False` uses all 12 videos. For the fair equal-cost comparison use
`fixed every 5 s` and `fixed every 6 s`, which bracket D's average rate (~5.6 s).

Results go to the same resumable file; section 9 analyses only the videos of this run.

In [ ]:
# QUICK = True: only the 4 TEST videos (204 questions); False: all 12 videos (834 questions)
QUICK = True
CONDITIONS = ['fixed every 3 s', 'D: full']
# also available: 'fixed every 4 s', 'fixed every 5 s', 'fixed every 6 s', 'fixed every 8 s',
#   'uniform (D budget)', 'hybrid 50/50 (D budget)', 'blind (no frames)', 'E: full',
#   'uniform (E budget)', 'D: ground truth', 'random (D budget)'
missing = [c for c in CONDITIONS if c not in sel]
assert not missing, f'unknown condition names {missing}; available: {list(sel)}'

RUN_VIDEOS = sorted(set(videos) & test_videos) if QUICK else videos
qs_run = qs[qs.video.isin(RUN_VIDEOS)].reset_index(drop=True)
print(f"{len(qs_run)} questions on {len(RUN_VIDEOS)} videos x {len(CONDITIONS)} conditions")
print(pd.DataFrame({c: {v: len(sel[c][v]) for v in RUN_VIDEOS} for c in CONDITIONS})
      .assign(video_s=[n_frames[v] for v in RUN_VIDEOS]).rename_axis("frames sent per video"))

OUT = os.path.join(VLM_DIR, f"{MODEL_ID.split('/')[-1]}_px{int(MAX_PIXELS**0.5)}_all12.jsonl")
qwen_mcq.run(model, proc, letter_ids, qs_run, sel, metas, OUT, CONDITIONS, MAX_PIXELS, tag=MODEL_ID)

## 9. Results

- **Accuracy per condition**, 95% interval from resampling the 12 videos.
- **Paired comparisons** on the same questions (`diff` = a − b in accuracy points;
  `a_only_right` / `b_only_right` = questions only one condition got right; `p_mcnemar`).
- **Accuracy by target coverage** (a frame within 2 s of the keyframe sent or not).
- **Per video**, so one video with many questions cannot drive the result unseen, and
  **test vs held-out training videos** separately.

In [ ]:
res = egoev.load_results(OUT)
bad = res[res['error'].notna()] if 'error' in res else res.iloc[:0]
if len(bad):
    print(f'{len(bad)} questions not scored (oom/nan), left out below:')
    display(bad.groupby(['condition', 'error']).size())
res = res[res['error'].isna()].copy() if 'error' in res else res.copy()
res['correct'] = res['correct'].astype(int)
res = res[res.video.isin(RUN_VIDEOS)]                # only the videos of this run (section 8)
res['split'] = np.where(res.video.isin(test_videos), 'test', 'train (held out)')

acc = egoev.accuracy_table(res)
display(acc.round(2))

pairs = [('D: full', f'fixed every {e} s') for e in (3, 4, 5, 6, 8)] + [
         ('D: full', 'uniform (D budget)'), ('hybrid 50/50 (D budget)', 'uniform (D budget)'),
         ('E: full', 'uniform (E budget)'), ('D: ground truth', 'fixed every 5 s'),
         ('fixed every 5 s', 'blind (no frames)')]
have = set(res.condition)
pt = pd.DataFrame([egoev.paired_test(res, a, b) for a, b in pairs if a in have and b in have])
print('paired, same questions (diff = a - b, accuracy points):')
display(pt.round(3))

m = res.merge(cov[['condition', 'qid', 'hit_2s']], on=['condition', 'qid'], how='left')
by_cov = m[m.condition != 'blind (no frames)'].groupby(['condition', 'hit_2s']).correct.agg(['mean', 'size'])
by_cov['mean'] *= 100
print('accuracy (%) when a frame within 2 s of the keyframe was / was not sent:')
display(by_cov.round(1).unstack())

print('accuracy (%) per video:')
pv = res.pivot_table(index='video', columns='condition', values='correct', aggfunc='mean') * 100
pv.insert(0, 'questions', res.groupby('video').qid.nunique())
display(pv.round(1))
print('accuracy (%) on test vs held-out training videos:')
display((res.pivot_table(index='split', columns='condition', values='correct', aggfunc='mean') * 100).round(1))

fig, ax = plt.subplots(figsize=(9, 5))
order = acc.sort_values('accuracy').index
ax.barh(order, acc.loc[order, 'accuracy'],
        xerr=[acc.loc[order, 'accuracy'] - acc.loc[order, 'ci_low'],
              acc.loc[order, 'ci_high'] - acc.loc[order, 'accuracy']], color='tab:blue', alpha=.8)
ax.axvline(20, ls=':', c='k', lw=1, label='chance (20%)')
ax.set_xlabel('accuracy (%)'); ax.set_title(f'{MODEL_ID}, {len(qs)} questions, {len(videos)} videos')
ax.legend(); plt.tight_layout(); plt.show()

# accuracy against frame rate: the fixed intervals trace what a fixed interval buys at each
# rate; D is a point. D beats fixed-interval sampling only if it sits above that line.
bt = egoev.budget_table(sel, n_frames)
pts = acc.join(bt['sends_per_min'], how='left')
fx = pts[pts.index.str.startswith('fixed every')].sort_values('sends_per_min')
fig, ax = plt.subplots(figsize=(8, 5))
if len(fx):
    ax.errorbar(fx['sends_per_min'], fx['accuracy'],
                yerr=[fx['accuracy'] - fx['ci_low'], fx['ci_high'] - fx['accuracy']],
                fmt='-o', color='0.5', capsize=3, label='fixed interval (one interval for all videos)')
    for name, r in fx.iterrows():
        ax.annotate(name.replace('fixed every ', ''), (r['sends_per_min'], r['accuracy']),
                    xytext=(4, 4), textcoords='offset points', fontsize=8, color='0.4')
for name, col in (('D: full', 'tab:blue'), ('uniform (D budget)', 'tab:orange'),
                  ('hybrid 50/50 (D budget)', 'tab:purple'), ('E: full', 'tab:green')):
    if name in pts.index:
        r = pts.loc[name]
        ax.errorbar([r['sends_per_min']], [r['accuracy']],
                    yerr=[[r['accuracy'] - r['ci_low']], [r['ci_high'] - r['accuracy']]],
                    fmt='s', color=col, capsize=3, label=name)
ax.set_xlabel('frames sent per minute of video'); ax.set_ylabel('accuracy (%)')
ax.set_title('accuracy vs frame rate: above the grey line = better than a fixed interval')
ax.grid(alpha=.3); ax.legend(fontsize=8); plt.tight_layout(); plt.show()

---

## Reading it

| look at | meaning |
|---|---|
| **hybrid vs uniform (D budget)** | does adding the gate's settle frames to an evenly spaced base beat even spacing alone at equal cost? the main question now |
| D vs uniform | whether the pilot's −4.4 points holds on 4× the questions |
| E vs uniform (E budget) | whether the timeout fixes D's coverage problem |
| D: ground truth vs uniform | the policy's ceiling; if it also loses, the policy design (not the model) is the limit |
| uniform vs blind | how much the frames help this VLM at all |
| coverage `within_2s` | the hybrid should cover nearly as many keyframes as uniform |
| per video / split | no single video or split should carry the result |

## Caveats

- Qwen2.5-VL-3B at 448 px: absolute accuracy is low; the comparison between conditions is
  what matters.
- 12 videos: intervals are still wide, but far narrower than the pilot's 4.
- The 8 training videos use held-out fold predictions, which are slightly noisier than the
  test videos' 5-fold ensemble.